In [5]:
# Parche defensivo: algunas etapas devuelven '-' en velocidad media y
# la implementación original lanza ValueError al convertir a float.
from procyclingstats import Stage

_avg_speed_winner_original = Stage.avg_speed_winner

def _avg_speed_winner_safe(self):
    try:
        return _avg_speed_winner_original(self)
    except (ValueError, TypeError):
        return None

Stage.avg_speed_winner = _avg_speed_winner_safe

In [1]:
import polars as pl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl import Workbook,utils
from openpyxl.drawing.image import Image
from procyclingstats import Race, Rider, Stage, RaceClimbs
import gc
import matplotlib.pyplot as plt
import numpy as np
import os
import json

# Función para insertar DataFrame en Excel con formato
def insertar_dataframe_en_excel(hojas, df_pandas,row, start_col=1):
    """
    Inserta encabezados y datos de un DataFrame en la hoja de Excel con formato.

    Args:
        hojas: Worksheet de openpyxl
        df_pandas: DataFrame en pandas para insertar
        row: Fila inicial para insertar el DataFrame
        start_col: Columna inicial para insertar el DataFrame (por defecto 1)
    """
    # Insertar encabezados del DataFrame
    for col_idx, col_name in enumerate(df_pandas.columns, start=start_col):
        cell = hojas.cell(row=row, column=col_idx, value=col_name)
        cell.font = Font(bold=True)
        cell.fill = PatternFill(start_color='D3D3D3', end_color='D3D3D3', fill_type='solid')
        cell.alignment = Alignment(horizontal='center')
    
    # Insertar datos del DataFrame
    for row_idx, row in enumerate(df_pandas.values, start=row + 1):
        for col_idx, value in enumerate(row, start=start_col):
            hojas.cell(row=row_idx, column=col_idx, value=value)

# gráfico radial de especialidades
def pintar_grafico_en_excel(hojas, df_especialidades_completo, row, column, df_especialidades_bbh=None, ancho=600, alto=600,year=2025):
    """
    Crea un gráfico radial (general vs opcional Burgos BH) y lo inserta en Excel.

    Args:
        hojas: Worksheet de openpyxl donde insertar el gráfico
        df_especialidades_completo: DataFrame de Polars con las especialidades generales
        row: Fila donde insertar la esquina superior de la imagen
        column: Columna donde insertar la esquina superior de la imagen
        df_especialidades_bbh: DataFrame de Polars para Burgos BH (opcional)
        ancho: Ancho de la imagen en píxeles (por defecto 600)
        alto: Alto de la imagen en píxeles (por defecto 600)
    """
    if df_especialidades_completo is None or df_especialidades_completo.is_empty():
        print("⚠️ No hay datos de especialidades para crear el gráfico")
        return
    
    # Columnas de especialidades (excluir rider_name, edition, position)
    cols_general = [c for c in df_especialidades_completo.columns if c not in ['rider_name', 'edition', 'position']]
    cols_bbh = [c for c in df_especialidades_bbh.columns if c not in ['rider_name', 'edition', 'position']] if df_especialidades_bbh is not None else []
    if not cols_general and not cols_bbh:
        print("⚠️ No hay columnas de especialidades para graficar")
        return
    
    # Unión de categorías ordenada alfabéticamente
    categories = sorted(cols_general + [c for c in cols_bbh if c not in cols_general])
    
    def mean_for(df, col):
        return df.select(pl.col(col)).fill_null(0).mean().item() if col in df.columns else 0.0
    
    medias_general = [mean_for(df_especialidades_completo, c) for c in categories]
    medias_bbh = [mean_for(df_especialidades_bbh, c) for c in categories] if df_especialidades_bbh is not None else None
    
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    
    general_plot = medias_general + [medias_general[0]]
    bbh_plot = medias_bbh + [medias_bbh[0]] if medias_bbh is not None else None
    max_val = max(general_plot + (bbh_plot or [])) if (general_plot + (bbh_plot or [])) else 1
    
    # Crear figura
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
    
    # Graficar general
    ax.plot(angles, general_plot, 'o-', linewidth=3, label='Media general', color='#FF5733')
    ax.fill(angles, general_plot, alpha=0.25, color='#FF5733')
    
    # Graficar Burgos BH si existe
    if bbh_plot is not None:
        ax.plot(angles, bbh_plot, 'o-', linewidth=3, label='Media Burgos BH', color='#1f77b4')
        ax.fill(angles, bbh_plot, alpha=0.2, color='#1f77b4')
        
        # Puntos individuales por ciclista Burgos BH
        df_bbh_pd = df_especialidades_bbh.select(categories + ['rider_name']).to_pandas()
        for idx, row_rider in df_bbh_pd.iterrows():
            vals = [row_rider.get(c, 0.0) if row_rider.get(c) is not None else 0.0 for c in categories]
            vals_loop = vals + [vals[0]]
            ax.plot(angles, vals_loop, marker='o', linestyle='', markersize=4, alpha=0.8, label=row_rider['rider_name'])
    
    # Configurar etiquetas
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=10, fontweight='bold')
    ax.set_ylim(0, max_val * 1.2 if max_val > 0 else 1)
    
    # Añadir cuadrícula
    ax.grid(True, linestyle='--', alpha=0.7)
    
    # Añadir título
    titulo = 'Perfil Medio de Especialidades' if bbh_plot is None else 'Perfil Medio: General vs Burgos BH'
    plt.title(titulo, size=14, y=1.08, fontweight='bold')
    if bbh_plot is not None:
        plt.legend(loc='upper right', bbox_to_anchor=(1.35, 1.05), fontsize=8)
    
    plt.tight_layout()
    
    # Guardar el gráfico como imagen temporal
    temp_image = f'temp/temp_grafico_especialidades_{year}.png'
    plt.savefig(temp_image, dpi=100, bbox_inches='tight', facecolor='white')
    plt.close()
    
    # Insertar imagen en Excel
    img = Image(temp_image)
    img.width = ancho
    img.height = alto
    celdax = utils.get_column_letter(column)
    celda = f"{celdax}{row}"
    hojas.add_image(img, celda)
    
  
#enlace_o_valor="/race/clasica-de-almeria/2026"
enlace_o_valor="/race/faun-ardeche-classic/2026"
nombre_archivo = f"data/{enlace_o_valor.replace('/', '_')}.json"
race = Race(f"{enlace_o_valor}/overview")
data = pl.DataFrame(race.parse())

'''if not os.path.exists(nombre_archivo):
    print(f"El archivo {nombre_archivo} no existe. Obteniendo datos de la web...")
    race = Race(f"{enlace_o_valor}/overview")
    data = race.parse()
    # Guardar en archivo JSON
    with open(nombre_archivo, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    print(f"El archivo {nombre_archivo} ya existe. Cargando datos...")
   
# esto para carreras de un solo día  
with open(nombre_archivo, 'r', encoding='utf-8') as f:
    data = json.load(f) 
print(data)
'''
file = f"data/HISTORIA_{data['name'].replace(' ', '_')}.xlsx"
libros=Workbook()
hojas=libros.active

# Insertar logo en A1
logo = Image('LOGO.png')
logo.width = 250
logo.height =100
hojas.add_image(logo, 'A1')

df_fin = None
#list_editions = race.prev_editions_select() 
c=hojas.max_row
df_especialidades_todos_bbh = []
# DataFrame para almacenar todas las especialidades
df_especialidades_todos = []
# DataFrame para especialidades de Burgos BH
df_especialidades_todos_bbh = []

hojas.cell(row=3, column=3, value=str(data['uci_tour'])+" "+data['name']+":"+str(data['startdate'])) # type: ignore
hojas['C3'].font = Font(color='FF0000', bold=True,size=25)    
c=hojas.max_row+2
#recorremos las 5 ultimas ediciones
#miramos si existen resultados previos
# Acumular datos de todas las ediciones
data_acumulado = {}
edicion=f"data/Resultados_{enlace_o_valor.strip()[0:len(enlace_o_valor)-4].replace('/', '_')}.json"
if not os.path.exists(edicion):
        print(f"El archivo {edicion} no existe. Obteniendo datos de la web...")
        for res in data['prev_editions_select'][0:6]:
            print(f"  Obteniendo datos de {res['value']}...")
            stage = Stage(f"{enlace_o_valor.strip()[0:len(enlace_o_valor)-4]}{res['text']}/result")
            resultado = stage.parse()
            # Acumular datos en diccionario
            data_acumulado[res['text']] = resultado
        # Guardar en archivo JSON una sola vez después de acumular
        with open(edicion,'w', encoding='utf-8') as f:
            json.dump(data_acumulado, f, indent=4, ensure_ascii=False)
        print(f"El archivo {edicion} ya existe. Cargando datos...")
    #cargamos resultados
for edition in data['prev_editions_select'][0:6]:
    c=hojas.max_row+1
    print("Procesando edición", edition['text'])
    #buscar puertos de la edición
    race_climbs = RaceClimbs(f"{enlace_o_valor.strip()[0:len(enlace_o_valor)-4]}{edition['text']}/route/climbs")

    # Transformar lista de climbs() a DataFrame
    climbs_list = race_climbs.climbs()
    df_climbs_list = pl.DataFrame(climbs_list)

    # Solo procesar si hay datos y existe la columna esperada
    if not df_climbs_list.is_empty() and 'km_before_finnish' in df_climbs_list.columns:
        df_climbs_list = df_climbs_list.sort('km_before_finnish', descending=True)
        # Insertar DataFrame de puertos en Excel
        insertar_dataframe_en_excel(
            hojas,
            df_climbs_list.select([
                pl.col('climb_name').alias('Puerto'),
                pl.col('length').alias('Longitud'),
                pl.col('steepness').alias('Desnivel'),
                pl.col('top').alias('Altitud'),
                pl.col('km_before_finnish').alias('km a meta')
            ]).to_pandas(),
            c + 4
        )
    else:
        hojas.cell(row=c+4, column=1, value="Sin datos de puertos para esta edición")

    cell=hojas.cell(row=c+2, column=1, value=f"Procesando edición {edition['text']}")
    cell.font = Font(bold=True,size=15)
    past_edit=Stage(f"{enlace_o_valor.strip()[0:len(enlace_o_valor)-4]}{edition['text']}/result")
    cell = hojas.cell(row=c+3, column=1, value=f'{past_edit.won_how()}')
    cell.font = Font(color='FF0000',bold=True)
    hojas.cell(row=c+3, column=2, value=f'participacion:{past_edit.race_startlist_quality_score()}' )
    hojas.cell(row=c+3, column=3, value=f'desnivel: {past_edit.vertical_meters()}m')
    hojas.cell(row=c+3, column=4, value=f'distancia {past_edit.distance()}km' )
    hojas.cell(row=c+3, column=5, value=f'avg:{past_edit.avg_speed_winner()}km/h')
    
    df_res = pl.DataFrame(past_edit.parse()['results'])
        
    if(df_res.is_empty()):
        print("  No hay resultados para esta edición.")
        hojas.cell(row=c+3, column=1, value="  No hay resultados para esta edición.")
        continue
    
    # Obtener especialidades para cada rider
    especialidades = []
    for idx, txirrindu in enumerate(df_res.head(10).select('rider_url', 'rider_name').iter_rows()):      
        rider_url, rider_name = txirrindu
        try:
            rider=Rider(str(rider_url)) # type: ignore 
        
            points_per_speciality = rider.parse()['points_per_speciality']
            if isinstance(points_per_speciality, dict):
                sorted_by_values = dict(sorted(points_per_speciality.items(), key=lambda item: item[1], reverse=True))
                df_rider_specialidad = pl.DataFrame([sorted_by_values])
            else:
                df_rider_specialidad = pl.DataFrame(points_per_speciality) 
        
            # Agregar información del rider y edición al DataFrame de especialidades
            df_rider_specialidad = df_rider_specialidad.with_columns([
                pl.lit(rider_name).alias('rider_name'),
                #pl.lit(edition['text']).alias('edition'),
                pl.lit(idx + 1).alias('position')
            ])
            df_especialidades_todos.append(df_rider_specialidad)
           
            cols = list(df_rider_specialidad.columns)
            row_values = df_rider_specialidad.row(0)
            # Buscar las columnas que no son las que agregamos
            cols_especialidad = [c for c in cols if c not in ['rider_name', 'edition', 'position']]
            if len(cols_especialidad) >= 2:
                especialidad = cols_especialidad[0]+":"+str(row_values[cols.index(cols_especialidad[0])])+","+cols_especialidad[1]+":"+str(row_values[cols.index(cols_especialidad[1])])
            elif len(cols_especialidad) == 1:
                especialidad = cols_especialidad[0]+":"+str(row_values[cols.index(cols_especialidad[0])])
            else:
                especialidad = "sin datos"
            
        except Exception as e:
            print(f"⚠️ Error procesando rider {rider_name}: {str(e)}")
            especialidades.append("Error al obtener datos")
        especialidades.append(especialidad)
        
    
    # Agregar columna de especialidad a df_res
    if len(especialidades) > 0:
        df_res_como = df_res.head(len(especialidades)).with_columns(pl.Series('especialidad', especialidades))
    
    # Insertar resultados generales (top 10)
    row_top10 = hojas.max_row + 2
    df_pandas = df_res_como.select([pl.col('rank').alias('Posicion'), pl.col('rider_name').alias('Nombre'), pl.col('time').alias('Tiempo'),pl.col('team_name').alias('Equipo'), pl.col('especialidad').alias('Especialidad')]).to_pandas()
    insertar_dataframe_en_excel(hojas, df_pandas, row_top10)
    
    # Insertar puntos UCI por equipos (a la derecha del top 10)
    df_pandas_uci=df_res.group_by('team_name').agg(pl.col('uci_points').sum()).filter(pl.col('uci_points') > 0).sort('uci_points', descending=True).to_pandas()
    df_pandas_uci = df_pandas_uci.rename(columns={'team_name': 'Equipo', 'uci_points': 'Puntos UCI'})
    
    insertar_dataframe_en_excel(hojas, df_pandas_uci, row_top10, start_col=7)

    # Insertar resultados del equipo Burgos
    df_burgos = df_res.filter(pl.col('team_name').str.contains('Burgos', strict=False))
    df_pandas = df_burgos.select([pl.col('rank').alias('Posicion'), pl.col('rider_name').alias('Nombre'), pl.col('time').alias('Tiempo'), pl.col('breakaway_kms').alias('Kms en fuga'), pl.col('uci_points').alias('Puntos UCI  ')]).to_pandas()
    insertar_dataframe_en_excel(hojas, df_pandas,row_top10+14)
   
    #preparando especialidades de Burgos BH para gráfico separado
   
    # Obtener especialidades para cada rider bbh
    especialidadesbbh = []
    especialidadbh = ""
    for idx, bagos in enumerate(df_burgos.select('rider_url', 'rider_name').iter_rows()):      
        try:
            rider_url, rider_name = bagos
            riderbbh=Rider(str(rider_url)) # type: ignore 
            points_per_speciality = riderbbh.parse()['points_per_speciality']
            if isinstance(points_per_speciality, dict):
                sorted_by_values = dict(sorted(points_per_speciality.items(), key=lambda item: item[1], reverse=True))
                df_rider_specialidad = pl.DataFrame([sorted_by_values])
            else:
                df_rider_specialidad = pl.DataFrame(points_per_speciality) 
        
            # Agregar información del rider y edición al DataFrame de especialidades
            df_rider_specialidad_bbh = df_rider_specialidad.with_columns([
            pl.lit(rider_name).alias('rider_name'),
            #pl.lit(edition['text']).alias('edition'),
            pl.lit(idx + 1).alias('position')
        ])
            df_especialidades_todos_bbh.append(df_rider_specialidad_bbh)
        
            cols = list(df_rider_specialidad_bbh.columns)
            row_values = df_rider_specialidad_bbh.row(0)
            # Buscar las columnas que no son las que agregamos
            cols_especialidad_bh = [c for c in cols if c not in ['rider_name', 'edition', 'position']]
        
            if len(cols_especialidad_bh) >= 2:
                especialidadbh = cols_especialidad_bh[0]+":"+str(row_values[cols.index(cols_especialidad_bh[0])])+","+cols_especialidad_bh[1]+":"+str(row_values[cols.index(cols_especialidad_bh[1])])
            elif len(cols_especialidad_bh) == 1:
                especialidadbh = cols_especialidad_bh[0]+":"+str(row_values[cols.index(cols_especialidad_bh[0])])
            else:
                especialidad = "sin datos"
        except Exception as e:
            print(f"⚠️ Error procesando rider {rider_name}: {str(e)}")
            especialidades.append("Error al obtener datos")
        especialidadesbbh.append(especialidadbh)
        
    
    # Agregar columna de especialidad a df_res
    if len(especialidadesbbh) > 0:
        df_res_como_bbh = df_res.head(len(especialidadesbbh)).with_columns(pl.Series('especialidad', especialidadesbbh))

    # Concatenar todos los DataFrames de especialidades
    if df_especialidades_todos:
        df_especialidades_completo = pl.concat(df_especialidades_todos, how='diagonal')
        print("\n📊 DataFrame de especialidades creado con", len(df_especialidades_completo), "riders")
   
    else:
        df_especialidades_completo = None
        print("\n⚠️ No se encontraron especialidades")

    # Concatenar especialidades de Burgos BH
    if df_especialidades_todos_bbh:
        df_especialidades_bbh_completo = pl.concat(df_especialidades_todos_bbh, how='diagonal')
        print("\n📊 DataFrame de especialidades Burgos BH creado con", len(df_especialidades_bbh_completo), "riders")
    
    else:
        df_especialidades_bbh_completo = None
        print("\n⚠️ No se encontraron especialidades de Burgos BH")

    # Insertar gráfico combinado en Excel
    
    pintar_grafico_en_excel(hojas, df_especialidades_completo, row_top10, 10, df_especialidades_bbh_completo, ancho=500, alto=400, year=edition['text'])
    
    # Vaciar las listas de especialidades después de escribir el gráfico en Excel
    df_especialidades_todos.clear()
    df_especialidades_todos_bbh.clear()
    
    gc.collect() 
# Ajustar ancho de columnas al contenido
for column_cells in hojas.columns:
    max_length = 0
    column_letter = utils.get_column_letter(column_cells[0].column)
    for cell in column_cells:
        if cell.value is not None:
           max_length = max(max_length, len(str(cell.value)))
    hojas.column_dimensions[column_letter].width = min(max_length + 2, 60)

libros.save(file)

# Eliminar carpeta temporal si existe



KeyboardInterrupt: 

In [2]:
# Mostrar puertos con información específica
print(f"Total de puertos: {len(df_climbs_list)}\n")
print(df_climbs_list.select(['climb_name', 'length', 'top','steepness','km_before_finnish']))

NameError: name 'df_climbs_list' is not defined

In [3]:
#insertar graficos combinados
if df_especialidades_completo is not None and df_especialidades_bbh_completo is not None:
    # Columnas de especialidades de cada dataset (excluyendo metadatos)
    cols_full = [c for c in df_especialidades_completo.columns if c not in ['rider_name', 'edition', 'position']]
    cols_bbh = [c for c in df_especialidades_bbh_completo.columns if c not in ['rider_name', 'edition', 'position']]
    if not cols_full or not cols_bbh:
        print('⚠️ No hay columnas de especialidades para combinar')
    else:
        # Mantener solo categorías presentes y ordenarlas alfabéticamente
        categories = sorted(cols_full + [c for c in cols_bbh if c not in cols_full])
        
        def mean_for(df, col):
            return df.select(pl.col(col)).fill_null(0).mean().item() if col in df.columns else 0.0
        
        medias_general = [mean_for(df_especialidades_completo, c) for c in categories]
        medias_bbh = [mean_for(df_especialidades_bbh_completo, c) for c in categories]
        
        N = len(categories)
        angles = [n / float(N) * 2 * np.pi for n in range(N)]
        angles += angles[:1]
        
        general_plot = medias_general + [medias_general[0]]
        bbh_plot = medias_bbh + [medias_bbh[0]]
        max_val = max(general_plot + bbh_plot) if (general_plot + bbh_plot) else 1
        
        fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
        ax.plot(angles, general_plot, 'o-', linewidth=3, label='Media general', color='#FF5733')
        ax.fill(angles, general_plot, alpha=0.25, color='#FF5733')
        ax.plot(angles, bbh_plot, 'o-', linewidth=3, label='Media Burgos BH', color='#1f77b4')
        ax.fill(angles, bbh_plot, alpha=0.2, color='#1f77b4')
        
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(categories, size=11)
        ax.set_ylim(0, max_val * 1.2 if max_val > 0 else 1)
        ax.grid(True)
        plt.title('Perfil Medio de Especialidades: General vs Burgos BH', size=16, y=1.08, fontweight='bold')
        plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
        plt.tight_layout()
        plt.show()
        
        print('\n📊 Media de puntos por especialidad (General):')
        for cat, val in zip(categories, medias_general):
            print(f'  {cat}: {val:.2f}')
        print('\n📊 Media de puntos por especialidad (Burgos BH):')
        for cat, val in zip(categories, medias_bbh):
            print(f'  {cat}: {val:.2f}')
else:
    print('⚠️ Faltan datos para generar el gráfico combinado (df_especialidades_completo o df_especialidades_bbh_completo)')


NameError: name 'df_especialidades_completo' is not defined

In [4]:
#carreras por etapas
enlace_o_valor="race/itzulia-basque-country/2026"
import gc
 

print("Intentando obtener resultados para:", enlace_o_valor)
race= Race(f"{enlace_o_valor}/overview")     
df_fin = None
list_editions = race.prev_editions_select() 
etapas=race.stages()
print(list_editions)

for edition in list_editions[1:10]:
   
    race= Race(f"{edition['value']}/overview")  
    print("========================================")
    print("Procesando edición", edition['text'])
    print("========================================")
    etapas=race.stages()
    for etapa in etapas:
        print(f"{etapa['stage_url']}/result")
        past_edit=Stage(f"{etapa['stage_url']}/result")
        #print(past_edit.parse().keys())
        print(past_edit.departure()+" - "+past_edit.arrival())
        print(f'como:{past_edit.won_how()} ,participacion:{past_edit.race_startlist_quality_score()},desnivel: {past_edit.vertical_meters()}m ,distancia {past_edit.distance()}km, avg:{past_edit.avg_speed_winner()}km/h')
        df_res = pl.DataFrame(past_edit.parse()['results'])
    
        #print(df_res['rider_url'].head(1))
        rider_url = df_res.select('rider_url').row(0)[0]
        rider=Rider(str(rider_url)) # type: ignore
    
        points_per_speciality = rider.parse()['points_per_speciality']
        if isinstance(points_per_speciality, dict):
            sorted_by_values = dict(sorted(points_per_speciality.items(), key=lambda item: item[1], reverse=True))
            df_rider_specialidad = pl.DataFrame([sorted_by_values])
        else:
            df_rider_specialidad = pl.DataFrame(points_per_speciality)
   
        cols = list(df_rider_specialidad.columns)
        especialidad = cols[0]+":"+str(df_rider_specialidad.row(0)[0])+","+cols[1]+":"+str(df_rider_specialidad.row(0)[1])
        print(especialidad)
        
        # Agregar columna de especialidad a df_res
        df_res = df_res.with_columns(pl.lit(especialidad).alias('especialidad'))
          
        print(df_res.select(['rank', 'rider_name','time','team_name', 'especialidad']).head(5))
        df = df_res.group_by('team_name').agg(pl.col('uci_points').sum()).sort('uci_points', descending=True)
        #print(df)
        df_burgos = df_res.filter(pl.col('team_name').str.contains('Burgos', strict=False))
     
        print(df_burgos.select(['rank', 'rider_name', 'time', 'breakaway_kms','uci_points', 'especialidad']).head(4))   
        gc.collect()

Intentando obtener resultados para: race/itzulia-basque-country/2026


NameError: name 'Race' is not defined

In [5]:
from playwright.async_api import async_playwright
def extraer_precios():
    with async_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()

        # Login automático
        page.goto("https://www.procyclingstat.com")
        #page.fill("#usuario", "mi_usuario")
        #page.fill("#password", "mi_clave")
        #page.click("button[type=submit]")
        print(page.title())
        # Esperar a que cargue el dashboard
        #page.wait_for_selector(".tabla-precios")

        

        browser.close()
        
extraer_precios()

TypeError: 'PlaywrightContextManager' object does not support the context manager protocol

In [6]:
import fitparse
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
fitfile = fitparse.FitFile('data/i117208346.fit')   
# Load the FIT f



#Get the headings from the activity file
allHeadings =[]

df = pl.DataFrame(fitfile)

for record in fitfile.get_messages("session"):
    for data in record:
        k = str(data.name) + ' (' + str(data.units) + ')'
        print(data.name + " @@@@ " + str(data.value))
        allHeadings.append(k)
    print("----- End of Record -----")
#Removes duplicates from headings
allHeadings = list(dict.fromkeys(allHeadings))


#Here we iterate over the records and format them into columns which are easier to plot.

records_list = []
for record in fitfile.get_messages("record"):
    dictR = {}
    for data in record:
        k = str(data.name) + ' (' + str(data.units) + ')'
        dictR[k] = data.value
        print(data.name + " :::::: " + str(data.value))
       
    records_list.append(dictR)
    print("----- End of Record -----")

df = pl.DataFrame(records_list)

print(df.columns)   
#Creates a column to give activity in seconds
##base_dt = df['timestamp (None)'].to_list()[0]
#df = df.with_columns((df['timestamp (None)'] - base_dt).dt.total_seconds().alias('Elapsed Time (s)'))


#Export to .CSV
print(df)
df.write_csv('output.csv') 
'''

#plt.plot(df['distance (m)'].to_list(), df['distance (m)'].to_list(), label='Distance')
#plt.plot(df['distance (m)'].to_list(), df['power (watts)'].to_list(), label='Pulso')
plt.plot(df['distance (m)'].to_list(), df['timestamp (None)'].to_list(), label='Power')
plt.plot(df['distance (m)'].to_list(), df['Elapsed Time (s)'].to_list(), label='Altitude')
plt.legend()
plt.show() '''

FileNotFoundError: [Errno 2] No such file or directory: 'data/i117208346.fit'

In [5]:
import cloudscraper
import procyclingstats as pcs

url = "https://www.strava.com/segments/1380959"
scraper = cloudscraper.create_scraper()
html_content: str
html_content = scraper.get(url).text

distancia=html_content.find("list-stats inline-stats stats-lg mt-md")
print("Posición de la sección de estadísticas:", distancia)


Posición de la sección de estadísticas: 39490


PRUEBAS POR ETAPAS

In [6]:
import polars as pl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl import Workbook,utils
from openpyxl.drawing.image import Image
from procyclingstats import Race, Rider, Stage, RaceClimbs
import gc
import matplotlib.pyplot as plt
import numpy as np
import os
import json
import time

def _obtener_con_reintentos(func, max_reintentos=2, espera_segundos=1):
    ultimo_error = None
    for intento in range(max_reintentos + 1):
        try:
            return func()
        except Exception as e:
            ultimo_error = e
            if intento < max_reintentos:
                print(f"⚠️ Reintento {intento + 1}/{max_reintentos} tras error: {e}")
                time.sleep(espera_segundos)
    print(f"❌ Falló tras {max_reintentos + 1} intentos: {ultimo_error}")
    return None

enlace_o_valor="/race/vuelta-a-la-comunidad-valenciana/2026"
print("Intentando obtener resultados para:", enlace_o_valor)
race= Race(f"{enlace_o_valor}/overview") 
    
data = race.parse()
stages = race.stages() 
     
for stage_info in stages:
    enlace_o_valor=stage_info.get('stage_url', 'desconocida')  
    print("Obteniendo etapa:",f"{enlace_o_valor}/result")      
    past_edit=Stage(f"{enlace_o_valor}/result")
    print("Procesando etapa:", past_edit.relative_url())
    print("Generando archivo Excel para la carrera:")
    '''
    # Usar la carpeta de destino proporcionada o la carpeta 'data/' por defecto
    if carpeta_destino:
        file = os.path.join(carpeta_destino, f"HISTORIA_{data['name'].replace(' ', '_')}.xlsx")
    else:'''
    file = f"data/HISTORIA_{data['name'].replace(' ', '_')}.xlsx"
    
    # DataFrame para acumular todos los top 10
    df_todos_top10 = []
    print(str(data['uci_tour'])+" "+data['name']+":"+str(data['startdate']))
    print("Archivo de salida:", file)
    libros=Workbook()
    hojas=libros.active
   
    #enlace_o_valor = race_slug
    # Insertar logo en A1
    logo = Image('LOGO.png')
    logo.width = 250
    logo.height =100
    hojas.add_image(logo, 'A1')

    df_fin = None
    #list_editions = race.prev_editions_select() 
    c=hojas.max_row
    df_especialidades_todos_bbh = []
    # DataFrame para almacenar todas las especialidades
    df_especialidades_todos = []
    # DataFrame para especialidades de Burgos BH
    df_especialidades_todos_bbh = []

    hojas.cell(row=3, column=3, value=str(data['uci_tour'])+" "+data['name']+":"+str(data['startdate'])) # type: ignore
    hojas['C3'].font = Font(color='FF0000', bold=True,size=25)    
    print("Ediciones encontradas:", len(data['prev_editions_select'] ))
    for res in data['prev_editions_select'][0:6]:
        print("Procesando edición:", res['text'])
        c=hojas.max_row+2
        
        cell=hojas.cell(row=c, column=1, value=f"Procesando edición {res['text']}")
        cell.font = Font(bold=True,size=15)
        print(f"{enlace_o_valor}/result") 
        past_edit=Stage(f"{enlace_o_valor}/result")
                
        cell = hojas.cell(row=c+2, column=1, value=f'{past_edit.won_how()}')
        cell.font = Font(color='FF0000',bold=True)
        hojas.cell(row=c+2, column=2, value=f'participacion:{past_edit.race_startlist_quality_score()}' )
        hojas.cell(row=c+2, column=3, value=f'desnivel: {past_edit.vertical_meters()} m')
        hojas.cell(row=c+2, column=4, value=f'distancia {past_edit.distance()} km' )
        #hojas.cell(row=c+2, column=5, value=f"avg:{past_edit.avg_speed_winner()} km/h")
        print("puerticos"+f"{enlace_o_valor}/route/climbs")
         #buscar puertos de la edición
        race_climbs = RaceClimbs(f"{enlace_o_valor}/route/climbs")
        # Transformar lista de climbs() a DataFrame
        climbs_list = race_climbs.climbs()
        df_climbs_list = pl.DataFrame(climbs_list)

        # Solo procesar si hay datos y existe la columna esperada
        if not df_climbs_list.is_empty() and 'km_before_finnish' in df_climbs_list.columns:
            df_climbs_list = df_climbs_list.sort('km_before_finnish', descending=True)
            # Insertar DataFrame de puertos en Excel
            insertar_dataframe_en_excel(
                hojas,
                df_climbs_list.select([
                    pl.col('climb_name').alias('Puerto'),
                    pl.col('length').alias('Longitud'),
                    pl.col('steepness').alias('Desnivel'),
                    pl.col('top').alias('Altitud'),
                    pl.col('km_before_finnish').alias('km a meta')
                ]).to_pandas(),
                c + 4
            )
        else:
            hojas.cell(row=c+4, column=1, value="Sin datos de puertos para esta edición")
        
        df_res = pl.DataFrame(past_edit.parse()['results'])
            
        if(df_res.is_empty()):
            print("  No hay resultados para esta edición.")
            hojas.cell(row=c+3, column=1, value="  No hay resultados para esta edición.")
            continue
        
        # Obtener especialidades para cada rider
        especialidades = []
        for idx, txirrindu in enumerate(df_res.head(10).select('rider_url', 'rider_name').iter_rows()):      
            if not txirrindu or len(txirrindu) < 2:
                print(f"⚠️ Datos incompletos en fila {idx}, saltando...")
                especialidades.append("sin datos")
                continue
            rider_url, rider_name = txirrindu
            try:
                # Obtener Rider con reintentos
                rider = _obtener_con_reintentos(
                    lambda ru=str(rider_url): Rider(ru),
                    max_reintentos=2
                )
                
                if rider is None:
                    print(f"⚠️ No se pudo obtener datos del ciclista {rider_name}")
                    especialidades.append("No disponible")
                    continue
            
                points_per_speciality = rider.parse()['points_per_speciality']
                if isinstance(points_per_speciality, dict):
                    sorted_by_values = dict(sorted(points_per_speciality.items(), key=lambda item: item[1], reverse=True))
                    df_rider_specialidad = pl.DataFrame([sorted_by_values])
                else:
                    df_rider_specialidad = pl.DataFrame(points_per_speciality) 
            
                # Agregar información del rider y edición al DataFrame de especialidades
                df_rider_specialidad = df_rider_specialidad.with_columns([
                    pl.lit(rider_name).alias('rider_name'),
                    #pl.lit(res['text']).alias('edition'),
                    pl.lit(idx + 1).alias('position')
                ])
                df_especialidades_todos.append(df_rider_specialidad)
            
                cols = list(df_rider_specialidad.columns)
                row_values = df_rider_specialidad.row(0)
                # Buscar las columnas que no son las que agregamos
                cols_especialidad = [c for c in cols if c not in ['rider_name', 'edition', 'position']]
                if len(cols_especialidad) >= 2:
                    especialidad = cols_especialidad[0]+":"+str(row_values[cols.index(cols_especialidad[0])])+","+cols_especialidad[1]+":"+str(row_values[cols.index(cols_especialidad[1])])
                elif len(cols_especialidad) == 1:
                    especialidad = cols_especialidad[0]+":"+str(row_values[cols.index(cols_especialidad[0])])
                else:
                    especialidad = "sin datos"
                
            except Exception as e:
                print(f"⚠️ Error procesando rider {rider_name}: {str(e)}")
                especialidad = "Error al obtener datos"
            
            especialidades.append(especialidad)
            # Limpiar memoria entre riders
            gc.collect()
            
        
        # Agregar columna de especialidad a df_res
        if len(especialidades) > 0:
            df_res_como = df_res.head(len(especialidades)).with_columns(pl.Series('especialidad', especialidades))
        
        # Insertar resultados generales (top 10)
        row_top10 = hojas.max_row + 2
        df_pandas = df_res_como.select([pl.col('rank').alias('Posicion'), pl.col('rider_name').alias('Nombre'), pl.col('time').alias('Tiempo'),pl.col('team_name').alias('Equipo'), pl.col('especialidad').alias('Especialidad')]).to_pandas()
        insertar_dataframe_en_excel(hojas, df_pandas, row_top10)
        
        # Agregar edición a los resultados para el treeview (mantener nombres originales)
        df_top10_edicion = df_res_como.select([
            'rank',
            'rider_name',
            'team_name',
            'time',
            'especialidad'
        ]).with_columns([
            pl.lit(res['text']).alias('edition')
        ])
        df_todos_top10.append(df_top10_edicion)
        
        # Insertar puntos UCI por equipos (a la derecha del top 10) - solo si la columna existe
        if 'uci_points' in df_res.columns:
            df_pandas_uci=df_res.group_by('team_name').agg(pl.col('uci_points').sum()).filter(pl.col('uci_points') > 0).sort('uci_points', descending=True).to_pandas()
            df_pandas_uci = df_pandas_uci.rename(columns={'team_name': 'Equipo', 'uci_points': 'Puntos UCI'})
            insertar_dataframe_en_excel(hojas, df_pandas_uci, row_top10, start_col=7)

        # Insertar resultados del equipo Burgos
        df_burgos = df_res.filter(pl.col('team_name').str.contains('Burgos', strict=False))
        # Seleccionar solo columnas disponibles
        cols_burgos = [pl.col('rank').alias('Posicion'), pl.col('rider_name').alias('Nombre'), pl.col('time').alias('Tiempo')]
        if 'breakaway_kms' in df_res.columns:
            cols_burgos.append(pl.col('breakaway_kms').alias('Kms en fuga'))
        if 'uci_points' in df_res.columns:
            cols_burgos.append(pl.col('uci_points').alias('Puntos UCI'))
        df_pandas = df_burgos.select(cols_burgos).to_pandas()
        insertar_dataframe_en_excel(hojas, df_pandas,row_top10+16)
    
        #preparando especialidades de Burgos BH para gráfico separado
    
        # Obtener especialidades para cada rider bbh
        especialidadesbbh = []
        especialidadbh = ""
        for idx, bagos in enumerate(df_burgos.select('rider_url', 'rider_name').iter_rows()):      
            if not bagos or len(bagos) < 2:
                print(f"⚠️ Datos incompletos en fila BBH {idx}, saltando...")
                especialidadesbbh.append("sin datos")
                continue
            try:
                rider_url, rider_name = bagos
                
                # Obtener Rider con reintentos
                riderbbh = _obtener_con_reintentos(
                    lambda ru=str(rider_url): Rider(ru),
                    max_reintentos=2
                )
                
                if riderbbh is None:
                    print(f"⚠️ No se pudo obtener datos del ciclista BBH {rider_name}")
                    especialidadbh = "No disponible"
                else:
                    points_per_speciality = riderbbh.parse()['points_per_speciality']
                    if isinstance(points_per_speciality, dict):
                        sorted_by_values = dict(sorted(points_per_speciality.items(), key=lambda item: item[1], reverse=True))
                        df_rider_specialidad = pl.DataFrame([sorted_by_values])
                    else:
                        df_rider_specialidad = pl.DataFrame(points_per_speciality) 
                
                    # Agregar información del rider y edición al DataFrame de especialidades
                    df_rider_specialidad_bbh = df_rider_specialidad.with_columns([
                    pl.lit(rider_name).alias('rider_name'),
                    #pl.lit(edition['text']).alias('edition'),
                    pl.lit(idx + 1).alias('position')
                ])
                    df_especialidades_todos_bbh.append(df_rider_specialidad_bbh)
                
                    cols = list(df_rider_specialidad_bbh.columns)
                    row_values = df_rider_specialidad_bbh.row(0)
                    # Buscar las columnas que no son las que agregamos
                    cols_especialidad_bh = [c for c in cols if c not in ['rider_name', 'edition', 'position']]
                
                    if len(cols_especialidad_bh) >= 2:
                        especialidadbh = cols_especialidad_bh[0]+":"+str(row_values[cols.index(cols_especialidad_bh[0])])+","+cols_especialidad_bh[1]+":"+str(row_values[cols.index(cols_especialidad_bh[1])])
                    elif len(cols_especialidad_bh) == 1:
                        especialidadbh = cols_especialidad_bh[0]+":"+str(row_values[cols.index(cols_especialidad_bh[0])])
                    else:
                        especialidadbh = "sin datos"
            except Exception as e:
                print(f"⚠️ Error procesando rider BBH {rider_name}: {str(e)}")
                especialidadbh = "Error al obtener datos"
            
            especialidadesbbh.append(especialidadbh)
            # Limpiar memoria entre riders
            gc.collect()
            
        
        # Agregar columna de especialidad a df_res
        if len(especialidadesbbh) > 0:
            df_res_como_bbh = df_res.head(len(especialidadesbbh)).with_columns(pl.Series('especialidad', especialidadesbbh))

        # Concatenar todos los DataFrames de especialidades
        if df_especialidades_todos:
            df_especialidades_completo = pl.concat(df_especialidades_todos, how='diagonal')
            print("\n📊 DataFrame de especialidades creado con", len(df_especialidades_completo), "riders")
    
        else:
            df_especialidades_completo = None
            print("\n⚠️ No se encontraron especialidades")

        # Concatenar especialidades de Burgos BH
        if df_especialidades_todos_bbh:
            df_especialidades_bbh_completo = pl.concat(df_especialidades_todos_bbh, how='diagonal')
            print("\n📊 DataFrame de especialidades Burgos BH creado con", len(df_especialidades_bbh_completo), "riders")
        
        else:
            df_especialidades_bbh_completo = None
            print("\n⚠️ No se encontraron especialidades de Burgos BH")

        # Insertar gráfico combinado en Excel
        # Asegurar texto de edición para el parámetro year
        edition_text = res['text'] if isinstance(res, dict) and 'text' in res else str(res)
        pintar_grafico_en_excel(hojas, df_especialidades_completo, row_top10, 10, df_especialidades_bbh_completo, ancho=500, alto=400, year=edition_text)
        
        # Vaciar las listas de especialidades después de escribir el gráfico en Excel
        df_especialidades_todos.clear()
        df_especialidades_todos_bbh.clear()
        print("fin for edición")
    
        gc.collect() 
    # Ajustar ancho de columnas al contenido
    # Ajustar ancho de columnas al contenido
    for column_cells in hojas.columns:
        max_length = 0
        column_letter = utils.get_column_letter(column_cells[0].column)
        for cell in column_cells:
            if cell.value is not None:
               max_length = max(max_length, len(str(cell.value)))
        hojas.column_dimensions[column_letter].width = min(max_length + 2, 45)
    
    # Guardar archivo Excel
    libros.save(file)
    print(f"✅ Archivo guardado: {file}")

Intentando obtener resultados para: /race/vuelta-a-la-comunidad-valenciana/2026
Obteniendo etapa: race/vuelta-a-la-comunidad-valenciana/2026/stage-1/result
Procesando etapa: race/vuelta-a-la-comunidad-valenciana/2026/stage-1/result
Generando archivo Excel para la carrera:
2.Pro Volta Comunitat Valenciana:2026-02-04
Archivo de salida: data/HISTORIA_Volta_Comunitat_Valenciana.xlsx
Ediciones encontradas: 77
Procesando edición: 2026
race/vuelta-a-la-comunidad-valenciana/2026/stage-1/result
puerticosrace/vuelta-a-la-comunidad-valenciana/2026/stage-1/route/climbs


AttributeError: 'NoneType' object has no attribute 'text'

puerticos de montaña 

In [6]:
from procyclingstats import Race, RaceClimbs, Stage
import polars as pl

enlace_o_valor = "/race/volta-a-catalunya/2025"
#palabra = "stage-2"  # cambia aquí la palabra a buscar en stage_url

print("Obteniendo puertos para:", f"{enlace_o_valor}")
race = Race(f"{enlace_o_valor}/overview")
race_climbs = RaceClimbs(f"{enlace_o_valor}/route/climbs")
stages = race.stages()
climbs_table = race_climbs.climbs()
carrerica=pl.DataFrame(race.prev_editions_select())
# mapear climb_url -> stage_url
climb_to_stage = {}
for stage_info in stages:
    stage_url = stage_info.get('stage_url', 'stage-1')
    stage = Stage(stage_url)
    for stage_climb in stage.climbs():
        climb_url = stage_climb.get('climb_url')
        if climb_url and climb_url not in climb_to_stage:
            climb_to_stage[climb_url] = stage_url

# añadir url_stage a cada puerto
enriched_climbs_table = [
    {**climb, 'url_stage': climb_to_stage.get(climb.get('climb_url'))}
    for climb in climbs_table
]

carrrr = pl.DataFrame(enriched_climbs_table)
print(carrrr.select(['url_stage', 'climb_name', 'length', 'top', 'steepness', 'km_before_finnish']))


# agrupar puertos por etapa y crear filas con stage_url + datos del puerto
for stage_info in stages:
    stage_url = stage_info.get('stage_url', 'stage-1')
    print(f"Procesando etapa: {stage_url}")
    stage = Stage(stage_url)

    stage_climbs = carrrr.filter(pl.col('url_stage') == stage_url).select([
        pl.col('climb_name'),
        pl.col('length'),
        pl.col('top'),
        pl.col('steepness'),
        pl.col('km_before_finnish')
    ])
    print(f"  Puertos encontrados: {stage_climbs}")
    





Obteniendo puertos para: /race/volta-a-catalunya/2025
shape: (18, 6)
┌─────────────────────────┬────────────────────────┬────────┬──────┬───────────┬───────────────────┐
│ url_stage               ┆ climb_name             ┆ length ┆ top  ┆ steepness ┆ km_before_finnish │
│ ---                     ┆ ---                    ┆ ---    ┆ ---  ┆ ---       ┆ ---               │
│ str                     ┆ str                    ┆ f64    ┆ i64  ┆ f64       ┆ i64               │
╞═════════════════════════╪════════════════════════╪════════╪══════╪═══════════╪═══════════════════╡
│ race/volta-a-catalunya/ ┆ Coll de la Creueta     ┆ 20.3   ┆ 1906 ┆ 5.1       ┆ 35                │
│ 2025/st…                ┆                        ┆        ┆      ┆           ┆                   │
│ race/volta-a-catalunya/ ┆ Coll d'Estenalles      ┆ 13.3   ┆ 843  ┆ 3.9       ┆ 162               │
│ 2025/st…                ┆                        ┆        ┆      ┆           ┆                   │
│ race/volta-a-catalun